# 05 – Apache Spark

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

Spark with the Iceberg runtime JAR is the most complete engine: DDL, `MERGE`, partition evolution,
branches/tags, maintenance procedures and v3 features (deletion vectors, row lineage, `VARIANT`).

The catalog's warehouse is attached as Spark catalog `sts`, on Lakekeeper also the
remote-signing warehouse as `signing` (the notebook then uses that one).
The first start downloads the Iceberg JARs from Maven (cached in `~/.ivy2`).

In [ ]:
from iceberg_lab.config import REMOTE_SIGNING, SETTINGS
from iceberg_lab.notebook import catalog_limit, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
from iceberg_lab.engines import spark_session

spark = spark_session()  # local[2], catalog "sts" (and "signing" where the catalog signs)
CAT = "signing" if REMOTE_SIGNING else "sts"
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CAT}.nb_spark")
spark.sql(f"SHOW NAMESPACES IN {CAT}").show()

<details><summary>Configuration set by <code>spark_session()</code> (example Lakekeeper)</summary>

```
spark.jars.packages = org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.12.0,
                      org.apache.iceberg:iceberg-aws-bundle:1.12.0
spark.sql.extensions = org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
spark.sql.catalog.signing = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.signing.type = rest
spark.sql.catalog.signing.uri = http://lakekeeper:8181/catalog
spark.sql.catalog.signing.warehouse = demo-signing
```
</details>

## DDL with hidden partitioning

> `DROP TABLE` here deliberately **without** `PURGE`: Lakekeeper deletes the data itself (`delete-profile: hard`), and Spark's client-side `PURGE` collides with that – on `demo-signing` the signer refuses deleting the already removed table, on `demo-sts` there is an intermittent `NoSuchKeyException` (see `tests/specific/test_spark.py::test_drop_table_purge`). On the other catalogs the files stay without `PURGE`: Polaris deletes only with purge, the RustFS catalog refuses purge (406), Unity deletes managed tables only after 7 days.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {CAT}.nb_spark.accounts")
spark.sql(f"""
    CREATE TABLE {CAT}.nb_spark.accounts (
        id BIGINT, owner STRING, balance DECIMAL(12,2), updated_at TIMESTAMP
    ) USING iceberg
    PARTITIONED BY (bucket(4, id))
    TBLPROPERTIES ('format-version'='2', 'write.merge.mode'='merge-on-read')
""")
spark.sql(f"""
    INSERT INTO {CAT}.nb_spark.accounts VALUES
      (1, 'ada', 100.00, current_timestamp()),
      (2, 'bob', 250.50, current_timestamp()),
      (3, 'cyd',   0.00, current_timestamp())
""")
spark.table(f"{CAT}.nb_spark.accounts").show()

## MERGE INTO (Upsert + Delete in einem Statement)

In [ ]:
spark.sql(f"""
    MERGE INTO {CAT}.nb_spark.accounts t
    USING (SELECT * FROM VALUES
             (2L, 'bob', CAST(300.00 AS DECIMAL(12,2))),
             (3L, 'cyd', CAST(NULL AS DECIMAL(12,2))),
             (4L, 'dan', CAST(42.00 AS DECIMAL(12,2))) AS s(id, owner, balance)) s
    ON t.id = s.id
    WHEN MATCHED AND s.balance IS NULL THEN DELETE
    WHEN MATCHED THEN UPDATE SET balance = s.balance, updated_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (id, owner, balance, updated_at)
                          VALUES (s.id, s.owner, s.balance, current_timestamp())
""")
spark.table(f"{CAT}.nb_spark.accounts").orderBy("id").show()

## Schema and partition evolution

In [ ]:
with catalog_limit("Partition evolution with metadata table"):
    spark.sql(f"ALTER TABLE {CAT}.nb_spark.accounts ADD COLUMN country STRING AFTER owner")
    spark.sql(f"ALTER TABLE {CAT}.nb_spark.accounts ADD PARTITION FIELD country")
    spark.sql(
        f"INSERT INTO {CAT}.nb_spark.accounts VALUES (5, 'eve', 'DE', 7.00, current_timestamp())"
    )
    spark.sql(
        f"SELECT spec_id, partition, record_count FROM {CAT}.nb_spark.accounts.partitions"
    ).show(truncate=False)

## Branches, Tags, Time Travel

In [ ]:
with catalog_limit("Tag and branch"):
    spark.sql(f"ALTER TABLE {CAT}.nb_spark.accounts CREATE TAG before_audit")
    spark.sql(f"ALTER TABLE {CAT}.nb_spark.accounts CREATE BRANCH audit")
    spark.sql(f"UPDATE {CAT}.nb_spark.accounts.branch_audit SET balance = 0")
    print("main:")
    spark.sql(f"SELECT sum(balance) FROM {CAT}.nb_spark.accounts").show()
    print("audit:")
    spark.sql(f"SELECT sum(balance) FROM {CAT}.nb_spark.accounts.branch_audit").show()
    print("tag:")
    spark.sql(f"SELECT count(*) FROM {CAT}.nb_spark.accounts VERSION AS OF 'before_audit'").show()

## Metadata tables and maintenance

In [ ]:
with catalog_limit("Metadata tables"):
    spark.sql(
        f"SELECT committed_at, snapshot_id, operation FROM {CAT}.nb_spark.accounts.snapshots"
    ).show(truncate=False)
    spark.sql(
        f"SELECT content, file_format, record_count FROM {CAT}.nb_spark.accounts.files"
    ).show()

In [ ]:
spark.sql(
    f"CALL {CAT}.system.rewrite_data_files(table => 'nb_spark.accounts', options => map('min-input-files','2'))"
).show()
spark.sql(f"CALL {CAT}.system.rewrite_position_delete_files(table => 'nb_spark.accounts')").show()
spark.sql(
    f"CALL {CAT}.system.expire_snapshots(table => 'nb_spark.accounts', retain_last => 3)"
).show()

## Format v3: Deletion Vectors, Row Lineage, VARIANT

In [ ]:
with catalog_limit("v3 table with VARIANT and deletion vector"):
    spark.sql("DROP TABLE IF EXISTS sts.nb_spark.events")
    spark.sql("CREATE NAMESPACE IF NOT EXISTS sts.nb_spark")
    spark.sql("""
        CREATE TABLE sts.nb_spark.events (id BIGINT, payload VARIANT) USING iceberg
        TBLPROPERTIES ('format-version'='3', 'write.delete.mode'='merge-on-read')
    """)
    spark.sql("""
        INSERT INTO sts.nb_spark.events VALUES
          (1, parse_json('{"type":"click","x":10}')),
          (2, parse_json('{"type":"view","tags":["a","b"]}')),
          (3, parse_json('{"type":"click","x":99}'))
    """)
    spark.sql("DELETE FROM sts.nb_spark.events WHERE id = 2")
    spark.sql("""
        SELECT id, variant_get(payload, '$.type', 'string') AS type,
               _row_id, _last_updated_sequence_number
        FROM sts.nb_spark.events ORDER BY id
    """).show()
    spark.sql(
        "SELECT content, file_format, record_count FROM sts.nb_spark.events.delete_files"
    ).show()